# Data Loading and Data Imputation Basics — Separate Hands-on Notebook

---

## Learning outcomes

By the end of this notebook, learners should be able to:

1. Create small sample datasets.
2. Save data into CSV files.
3. Load CSV data using Pandas.
4. Inspect missing values.
5. Understand why missing values matter.
6. Apply different imputation methods:
   - mean imputation,
   - median imputation,
   - mode imputation,
   - constant-value imputation,
   - forward fill,
   - backward fill.
7. Compare data before and after imputation.
8. Save cleaned data into a new CSV file.

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

# 1. Creating Sample Data

In a real project, data may come from:

- CSV file,
- Excel file,
- database,
- sensor logs,
- web API,
- manually collected observations.

Here, we first create a small student performance dataset with intentional missing values.

In [ ]:
student_data = pd.DataFrame({
    "Student_ID": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "Name": ["Asha", "Bharat", "Chitra", "Deepak", "Esha", "Farhan", "Gita", "Harish", "Iqra", "Jatin"],
    "Department": ["CSE", "ECE", "CSE", "ME", "ECE", "ME", "CSE", "ECE", "ME", "CSE"],
    "Math": [80, 60, 88, np.nan, 95, 55, 78, 66, np.nan, 72],
    "Physics": [75, 65, 92, 55, np.nan, np.nan, 82, 70, 60, 74],
    "Computer": [90, 70, np.nan, 50, 98, 65, 84, np.nan, 58, 80],
    "Attendance": [92, 81, 95, 70, 96, 75, np.nan, 88, 73, 90],
    "Result": ["Pass", "Pass", "Pass", "Fail", "Pass", "Pass", "Pass", "Pass", "Pass", "Pass"]
})

student_data

# 2. Saving Data into a CSV File

CSV means **Comma-Separated Values**.  
It is one of the most common formats for storing tabular data.

In [ ]:
student_data.to_csv("student_performance_missing.csv", index=False)

print("CSV file saved successfully.")

# 3. Loading Data from CSV

Use:

```python
pd.read_csv("filename.csv")
```

In [ ]:
df = pd.read_csv("student_performance_missing.csv")

df

# 4. Inspecting Loaded Data

Before cleaning the data, always inspect it.

In [ ]:
print("Shape of dataset:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nDataset information:")
df.info()

# 5. Checking Missing Values

Important functions:

- `isna()` detects missing values.
- `sum()` counts missing values column-wise.
- `isna().sum().sum()` counts total missing values.

In [ ]:
missing_per_column = df.isna().sum()
total_missing = df.isna().sum().sum()

print("Missing values per column:")
print(missing_per_column)

print("\nTotal missing values:", total_missing)

# 6. Display Rows Having Missing Values

This is useful to manually inspect incomplete records.

In [ ]:
rows_with_missing = df[df.isna().any(axis=1)]

rows_with_missing

# 7. Visualizing Missing Values

A simple bar chart helps identify which columns need cleaning.

In [ ]:
missing_per_column.plot(kind="bar", title="Missing Values per Column")
plt.xlabel("Columns")
plt.ylabel("Number of Missing Values")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 8. Method 1: Dropping Missing Values

`dropna()` removes rows or columns containing missing values.

This is simple, but it can reduce dataset size.  
Use it carefully, especially with small datasets.

In [ ]:
df_dropped_rows = df.dropna()

print("Original shape:", df.shape)
print("After dropping rows:", df_dropped_rows.shape)

df_dropped_rows

# 9. Method 2: Mean Imputation

Mean imputation replaces missing values with the column mean.

It is commonly used for numerical columns, but it is sensitive to outliers.

In [ ]:
df_mean = df.copy()

numeric_columns = ["Math", "Physics", "Computer", "Attendance"]

for col in numeric_columns:
    mean_value = df_mean[col].mean()
    df_mean[col] = df_mean[col].fillna(mean_value)
    print(f"{col}: missing values filled with mean = {mean_value:.2f}")

df_mean

# 10. Method 3: Median Imputation

Median imputation replaces missing values with the column median.

Median is often better than mean when data contains outliers.

In [ ]:
df_median = df.copy()

for col in numeric_columns:
    median_value = df_median[col].median()
    df_median[col] = df_median[col].fillna(median_value)
    print(f"{col}: missing values filled with median = {median_value:.2f}")

df_median

# 11. Method 4: Mode Imputation for Categorical Data

Mode means the most frequent value.

It is useful for categorical columns such as:

- department,
- gender,
- class label,
- device status,
- diagnosis group.

In [ ]:
# Example categorical data with missing values
category_df = pd.DataFrame({
    "Name": ["Asha", "Bharat", "Chitra", "Deepak", "Esha", "Farhan"],
    "Department": ["CSE", "ECE", np.nan, "ME", "ECE", np.nan]
})

display(category_df)

mode_department = category_df["Department"].mode()[0]
category_df["Department"] = category_df["Department"].fillna(mode_department)

print("Mode department:", mode_department)
display(category_df)

# 12. Method 5: Constant-Value Imputation

Sometimes missing values are replaced by a fixed value.

Examples:

- missing category → `"Unknown"`
- missing sensor status → `"Not Recorded"`
- missing numerical value → `0`

This is useful when the missingness itself is meaningful.

In [ ]:
category_df = pd.DataFrame({
    "Name": ["Asha", "Bharat", "Chitra", "Deepak", "Esha", "Farhan"],
    "Department": ["CSE", "ECE", np.nan, "ME", "ECE", np.nan]
})

category_df["Department"] = category_df["Department"].fillna("Unknown")

category_df

# 13. Method 6: Forward Fill and Backward Fill

These methods are useful for time-series or sequential data.

- Forward fill: use previous available value.
- Backward fill: use next available value.

In [ ]:
sensor_data = pd.DataFrame({
    "Time_sec": list(range(1, 16)),
    "Temperature_C": [25.1, 25.3, np.nan, 25.8, 26.0, np.nan, 26.4, 26.5, 26.8, np.nan, 27.1, 27.2, 27.5, 27.6, 27.9],
    "Humidity_percent": [45, 46, 46, np.nan, 48, 49, np.nan, 51, 52, 53, np.nan, 54, 55, 56, 57],
    "Pressure_kPa": [101.2, 101.3, 101.3, 101.4, np.nan, 101.5, 101.6, 101.6, np.nan, 101.7, 101.8, 101.8, 101.9, np.nan, 102.0],
    "Device_Status": ["OK", "OK", "OK", "OK", "Warning", "OK", np.nan, "OK", "Warning", "OK", "OK", np.nan, "OK", "OK", "Warning"]
})

sensor_data.to_csv("sensor_readings_missing.csv", index=False)

sensor_df = pd.read_csv("sensor_readings_missing.csv")

sensor_df

In [ ]:
sensor_ffill = sensor_df.copy()
sensor_ffill = sensor_ffill.ffill()

sensor_bfill = sensor_df.copy()
sensor_bfill = sensor_bfill.bfill()

print("Forward-filled data:")
display(sensor_ffill)

print("Backward-filled data:")
display(sensor_bfill)

# 14. Interpolation for Numerical Time-Series Data

Interpolation estimates missing values between known values.

It is useful when values change smoothly over time, such as:

- temperature,
- humidity,
- voltage,
- pressure,
- biomedical signals.

In [ ]:
sensor_interpolated = sensor_df.copy()

numeric_sensor_columns = ["Temperature_C", "Humidity_percent", "Pressure_kPa"]

sensor_interpolated[numeric_sensor_columns] = sensor_interpolated[numeric_sensor_columns].interpolate(method="linear")

# For categorical column, use forward fill
sensor_interpolated["Device_Status"] = sensor_interpolated["Device_Status"].ffill()

sensor_interpolated

# 15. Comparing Original and Imputed Sensor Data

Let us compare temperature before and after interpolation.

In [ ]:
plt.plot(sensor_df["Time_sec"], sensor_df["Temperature_C"], marker="o", label="Original")
plt.plot(sensor_interpolated["Time_sec"], sensor_interpolated["Temperature_C"], marker="x", label="Interpolated")

plt.title("Temperature: Original vs Interpolated")
plt.xlabel("Time in seconds")
plt.ylabel("Temperature in °C")
plt.legend()
plt.tight_layout()
plt.show()

# 16. Recommended Practical Workflow

A good data-cleaning workflow is:

1. Load the dataset.
2. Inspect shape, columns, and data types.
3. Count missing values.
4. Decide column-wise imputation strategy.
5. Apply imputation.
6. Verify that missing values are removed.
7. Save cleaned data.

# 17. Complete Example: Student Dataset Cleaning Pipeline

For this dataset:

- Numerical columns: use median imputation.
- Categorical columns: use mode imputation.
- Then compute `Total` and `Average`.
- Save the cleaned data.

In [ ]:
clean_df = pd.read_csv("student_performance_missing.csv")

numeric_columns = ["Math", "Physics", "Computer", "Attendance"]
categorical_columns = ["Department", "Result"]

# Numerical imputation using median
for col in numeric_columns:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

# Categorical imputation using mode
for col in categorical_columns:
    clean_df[col] = clean_df[col].fillna(clean_df[col].mode()[0])

# Create useful derived columns
clean_df["Total"] = clean_df[["Math", "Physics", "Computer"]].sum(axis=1)
clean_df["Average"] = clean_df[["Math", "Physics", "Computer"]].mean(axis=1)

# Assign grade
conditions = [
    clean_df["Average"] >= 85,
    clean_df["Average"] >= 70,
    clean_df["Average"] >= 50
]

choices = ["A", "B", "C"]

clean_df["Grade"] = np.select(conditions, choices, default="D")

print("Missing values after cleaning:")
print(clean_df.isna().sum())

display(clean_df)

clean_df.to_csv("student_performance_cleaned.csv", index=False)

print("Cleaned data saved as student_performance_cleaned.csv")

# 18. Complete Example: Sensor Dataset Cleaning Pipeline

For this dataset:

- Use linear interpolation for numerical sensor values.
- Use forward fill for device status.
- Save the cleaned sensor dataset.

In [ ]:
clean_sensor_df = pd.read_csv("sensor_readings_missing.csv")

numeric_sensor_columns = ["Temperature_C", "Humidity_percent", "Pressure_kPa"]

clean_sensor_df[numeric_sensor_columns] = clean_sensor_df[numeric_sensor_columns].interpolate(method="linear")
clean_sensor_df["Device_Status"] = clean_sensor_df["Device_Status"].ffill()

print("Missing values after cleaning:")
print(clean_sensor_df.isna().sum())

display(clean_sensor_df)

clean_sensor_df.to_csv("sensor_readings_cleaned.csv", index=False)

print("Cleaned sensor data saved as sensor_readings_cleaned.csv")

# 19. Mini Practice

Try these exercises:

## Exercise 1
Load `student_performance_missing.csv` and fill missing numerical values using mean instead of median.

## Exercise 2
Load `sensor_readings_missing.csv` and fill missing numerical values using forward fill.

## Exercise 3
Compare mean, median, and forward-fill imputation results for the `Physics` column.

## Exercise 4
Create a function that accepts a DataFrame and returns missing-value count and percentage for each column.

In [ ]:
# Exercise 4 solution template

def missing_summary(dataframe):
    summary = pd.DataFrame({
        "Missing_Count": dataframe.isna().sum(),
        "Missing_Percentage": dataframe.isna().mean() * 100
    })
    return summary.sort_values(by="Missing_Count", ascending=False)

missing_summary(df)

# 20. Recap

In this notebook, we covered:

- creating sample datasets,
- saving CSV files,
- loading data using `pd.read_csv()`,
- checking missing values,
- visualizing missingness,
- dropping missing rows,
- mean imputation,
- median imputation,
- mode imputation,
- constant-value imputation,
- forward fill,
- backward fill,
- interpolation,
- complete cleaning pipelines,
- saving cleaned datasets.

This notebook can be used as a bridge between basic Pandas and machine learning preprocessing.